# Müdahaleci Shapley değerleri: kendi kodumuz, kütüphane, arka plan seçimi — NIJ 2021

Bu notebook, önceki haftaların devamıdır ve aynı veri kümesi üzerinde yürütülür (Georgia'da şartlı tahliye edilen bireyler, hedef: 3 yıl içinde yeniden tutuklanma). Açıklanan model, ikinci haftadaki kara kutu (gradient boosting) yapılandırmasıdır. Soru şudur: bir kişi için Shapley değerleri, **hangi arka plana göre** hesaplandığına ne kadar bağlıdır ve kendi yazdığımız kesin hesap kütüphaneyle aynı sonucu veriyor mu?

## Adımlar

0. Veri, model, oyuncu seçimi (en önemli 10 öznitelik), kişi seçimi
1. Kesin müdahaleci Shapley'nin kendi kodu ve dört aksiyom testi (verimlilik, kukla, simetri, toplamsallık)
2. `shap` kütüphanesiyle karşılaştırma (müdahaleci TreeSHAP, ExactExplainer, KernelExplainer)
3. Arka plan boyutu: 1, 10, 100, 500; 20 tohum; referanstan göreli sapma. Ek olarak arka plan bileşimi (yeniden tutuklanma oranı)
4. Anlamlı referans grupları: taban ve Shapley değerleri, işaret değiştiren öznitelikler
5. Yol bağımlı ve müdahaleci TreeSHAP karşılaştırması (ağaç modeli kullanıldığı için)

## Ortak tasarım kararları

**Model.** İkinci haftadaki `HistGradientBoostingClassifier`; hiperparametreler yeniden aranmaz, 15 dış katın hepsinde seçilen yapılandırma kullanılır. Tek fark girdi kodlamasıdır: TreeSHAP'ta bir oyuncu bir sütundur, one-hot kodlu bir kategorik özniteliğin sütunları ise ayrı oyuncu olurdu. Bu yüzden iki kategorik öznitelik (`Gang_Affiliated`, `Prison_Offense`) bu notebook'ta sabit bir düzey sırasıyla tamsayı kodlanır (sıra 0.1'deki hücrede yazılıdır) ve model aynı hiperparametrelerle bu girdiyle yeniden eğitilir. Düzeylerin doğal bir sırası olmadığı için bu kodlama ağaçlar açısından bir ödünleşimdir; etkisi, ikinci haftadaki dış kat AUC'si ile bu notebook'taki test AUC'sinin karşılaştırmasından görülür.

**Oyuncular.** 24 öznitelikten eğitim kümesindeki permütasyon önemine göre en önemli **10** tanesi oyuncudur (2¹⁰ = 1024 koalisyon, kesin hesap mümkün). Kalan 14 öznitelik, **eğitim kümesinin ortalamasına sabitlenir** (hem açıklanan kişide hem arka plan satırlarında); yani oyuna katılmazlar. Bu, açıklanan işlevin `g(z) = f(z, m)` olduğu anlamına gelir; `f` modelin log-odds çıktısı, `m` kalan 14 özniteliğin eğitim ortalamasıdır.

**Değer fonksiyonu.** Müdahaleci (marjinal): `v(S) = E_b[ g(x_S, b_(N∖S)) ]`; beklenti arka plan satırları `b` üzerindendir. Çıktı ölçeği log-odds'tur (`decision_function`). Taban `v(∅) = E_b[g(b)]`, toplam `v(N) = g(x)`.

**Izgara tablosu.** 10 oyuncunun hepsi ayrık olduğu için (en çok 11 düzey) tüm düzey kombinasyonlarında `g` bir kez hesaplanıp tabloya yazılır; sonraki bütün Shapley hesapları bu tablodan okunur. Tablonun modelle aynı değerleri verdiği ayrıca sınanır.

**Rastgelelik.** Tüm hesaplar `SEED = 42` ile tekrarlanabilir; arka plan deneylerinde her tohum ayrı bir rastgele altküme çeker.

In [ ]:
import json
import time
import warnings
from math import factorial

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

SEED = 42
TEST_SIZE = 0.30
N_PLAYERS = 10            # oyuncu sayısı (en önemli öznitelikler)
PI_REPEATS = 5            # oyuncu seçimi için permütasyon tekrarı (eğitim kümesinde)
SIZES = [1, 10, 100, 500]  # arka plan boyutları
N_SEEDS = 20              # her boyut için tohum sayısı (en az 5)
COMP_SIZE = 200           # bileşim deneyinde arka plan boyutu
COMP_FRACS = [0.0, 0.25, 0.5, 0.75, 1.0]   # arka plandaki yeniden tutuklanan (y=1) oranı
N_COMP_SEEDS = 20
B_SHAP = 100              # kütüphane karşılaştırmasında arka plan boyutu
B_PATH = 2000             # 5. adımda müdahaleci TreeSHAP arka plan boyutu
EPS_SIGN = 0.01           # işaret karşılaştırmasında önemsiz sayılan en küçük |φ| (log-odds)
DATA_CSV = "nij_clean.csv"
RESULT_CSV = "sonuc_dis_katlar.csv"

def savefig(fig, name):
    fig.savefig(name, dpi=150, bbox_inches="tight")

print("shap", shap.__version__)

## Adım 0 — Veri, model, oyuncular, kişiler

### 0.1 Veri ve model
Eğitim / test ayrımı dördüncü haftayla aynıdır (%70 / %30, hedefe göre tabakalı, tohum 42). Kategorik öznitelikler, aşağıda yazılı sabit bir sırayla tamsayı kodlanır.

In [ ]:
df = pd.read_csv(DATA_CSV)
y = df["y"].astype(int).to_numpy()
meta = df[["Race", "Gender"]]
X = df.drop(columns=["y", "Race", "Gender"]).copy()

ORDER = {
    "Gang_Affiliated": ["Unknown", "No", "Yes"],
    "Prison_Offense": ["Other", "Drug", "Property", "Unknown", "Violent/Non-Sex", "Violent/Sex"],
}
for c, lv in ORDER.items():
    assert set(X[c].unique()) == set(lv), f"{c}: beklenmeyen düzeyler {set(X[c].unique())}"
    X[c] = X[c].map({v: i for i, v in enumerate(lv)})

FEATS = list(X.columns)
Xn = X.to_numpy(float)
idx_tr, idx_te = train_test_split(np.arange(len(X)), test_size=TEST_SIZE, stratify=y, random_state=SEED)
Xtr, Xte = Xn[idx_tr], Xn[idx_te]
ytr, yte = y[idx_tr], y[idx_te]
meta_tr = meta.iloc[idx_tr].reset_index(drop=True)
meta_te = meta.iloc[idx_te].reset_index(drop=True)
print(f"Satır: {len(X):,} | öznitelik: {len(FEATS)} | eğitim: {len(Xtr):,} (y=1 oranı {ytr.mean():.3f}) | test: {len(Xte):,}")

res = pd.read_csv(RESULT_CSV)
cfg_gb = json.loads(res.loc[res["model"] == "GB", "best_cfg"].value_counts().index[0])
print("Kara kutu yapılandırması (ikinci hafta):", cfg_gb)

bb = HistGradientBoostingClassifier(early_stopping=False, random_state=SEED, **cfg_gb).fit(Xtr, ytr)
f = bb.decision_function          # log-odds
print(f"AUC — eğitim: {roc_auc_score(ytr, f(Xtr)):.4f} | test: {roc_auc_score(yte, f(Xte)):.4f}")
print(f"Karşılaştırma: ikinci haftada GB dış kat AUC ortalaması {res.loc[res['model'] == 'GB', 'auc'].mean():.4f}")

### 0.2 Oyuncu seçimi
En önemli 10 öznitelik, **eğitim kümesinde** permütasyon önemine (AUC düşüşü, 5 tekrar) göre seçilir; böylece seçim test kümesine bakmaz. Kalan 14 öznitelik eğitim ortalamasında sabitlenir. Sabitlemenin modelden ne kadar bilgi götürdüğü, `g` işlevinin test AUC'siyle gösterilir.

In [ ]:
def perm_imp_auc(M, yy, repeats, seed):
    rng = np.random.default_rng(seed)
    base = roc_auc_score(yy, f(M))
    M2 = M.copy()
    out = {}
    for j, name in enumerate(FEATS):
        orig = M[:, j].copy()
        d = []
        for _ in range(repeats):
            M2[:, j] = orig[rng.permutation(len(M))]
            d.append(base - roc_auc_score(yy, f(M2)))
        M2[:, j] = orig
        out[name] = float(np.mean(d))
    return pd.Series(out)

pi_tr = perm_imp_auc(Xtr, ytr, PI_REPEATS, SEED).sort_values(ascending=False)
SEL_NAMES = list(pi_tr.index[:N_PLAYERS])
SEL = [FEATS.index(n) for n in SEL_NAMES]
OTH = [j for j in range(len(FEATS)) if j not in SEL]
m_full = Xtr.mean(axis=0)

def to_full(Z):
    # Z: (n, N_PLAYERS) oyuncu değerleri -> (n, 24); kalan 14 öznitelik eğitim ortalamasında
    T = np.tile(m_full, (len(Z), 1))
    T[:, SEL] = Z
    return T

def g_model(Z):
    return f(to_full(Z))

print("Oyuncular (eğitim permütasyon önemi, AUC düşüşü):")
print(pi_tr.head(N_PLAYERS + 4).round(4).to_string())
print("\nSabitlenen öznitelikler:", [FEATS[j] for j in OTH])
print(f"\ng işlevinin test AUC'si: {roc_auc_score(yte, g_model(Xte[:, SEL])):.4f}  (tam model: {roc_auc_score(yte, f(Xte)):.4f})")

### 0.3 Izgara tablosu ve `g` işlevi
10 oyuncunun düzey sayıları çarpılarak ızgara boyutu bulunur. Her kombinasyonda model bir kez çalıştırılıp tabloya yazılır. Bundan sonra `g` değerleri tablodan okunur; kendi Shapley kodumuz bu tabloyu bir kara kutu işlevi gibi çağırır.

In [ ]:
LV = [np.unique(Xn[:, j]) for j in SEL]
SHAPE = tuple(len(v) for v in LV)
STR = np.array([int(np.prod(SHAPE[k + 1:])) for k in range(N_PLAYERS)], dtype=np.int64)
N_GRID = int(np.prod(SHAPE))
print("Oyuncu düzey sayıları:", dict(zip(SEL_NAMES, SHAPE)))
print(f"Izgara boyutu: {N_GRID:,} kombinasyon")
assert N_GRID <= 80_000_000, "ızgara çok büyük; N_PLAYERS değerini azaltın"

t0 = time.time()
TAB = np.empty(N_GRID)
CH = 400_000
for s in range(0, N_GRID, CH):
    idx = np.arange(s, min(s + CH, N_GRID))
    ix = np.unravel_index(idx, SHAPE)
    Z = np.column_stack([LV[k][ix[k]] for k in range(N_PLAYERS)])
    TAB[s:s + len(idx)] = g_model(Z)
    if (s // CH) % 10 == 0:
        print(f"  ızgara: {min(s + CH, N_GRID):,}/{N_GRID:,}  ({time.time() - t0:.0f} sn)")
print(f"Izgara tablosu hazır ({time.time() - t0:.0f} sn)")

def to_ix(Z):
    # değerleri düzey sıra numaralarına çevirir (ızgara indeksi)
    IX = np.column_stack([np.searchsorted(LV[k], Z[:, k]) for k in range(N_PLAYERS)]).astype(np.int64)
    back = np.column_stack([LV[k][IX[:, k]] for k in range(N_PLAYERS)])
    assert np.array_equal(back, Z), "değer ızgarada yok"
    return IX

def g_ix(IX):
    # indeks uzayında g: ızgara tablosundan okur
    return TAB[(IX * STR).sum(axis=-1)]

IXtr, IXte = to_ix(Xtr[:, SEL]), to_ix(Xte[:, SEL])
chk = np.random.default_rng(SEED).choice(len(Xtr), min(5000, len(Xtr)), replace=False)
print("Tablo ile modelin en büyük mutlak farkı (5000 eğitim satırı):", np.abs(g_ix(IXtr[chk]) - g_model(Xtr[chk][:, SEL])).max())

### 0.4 Üç kişi
Kişiler **test kümesinden** ve oyunun çıktısına (`g`) göre seçilir: yüksek riskli bir erkek (g olasılığının test dağılımındaki %90'lık dilimine en yakın), orta riskli bir kadın (%50) ve düşük riskli bir erkek (%10). Aynı üç kişi bütün deneylerde kullanılır.

In [ ]:
sig = lambda s: 1 / (1 + np.exp(-s))
g_tr, g_te = g_ix(IXtr), g_ix(IXte)
gp_te = sig(g_te)
gender_te = meta_te["Gender"].to_numpy()

def pick(q, gender):
    t = np.quantile(gp_te, q)
    ok = np.where(gender_te == gender)[0]
    return int(ok[np.argmin(np.abs(gp_te[ok] - t))])

PERSONS = {"Kişi A": pick(0.90, "M"), "Kişi B": pick(0.50, "F"), "Kişi C": pick(0.10, "M")}
PX = {n: IXte[i] for n, i in PERSONS.items()}      # indeks uzayı (10,)
PXV = {n: Xte[i][SEL] for n, i in PERSONS.items()}  # değer uzayı (10,)

def decode(name, v):
    if name in ORDER:
        return ORDER[name][int(v)]
    return f"{v:g}"

rows = []
for n, i in PERSONS.items():
    r = {"kişi": n, "test satırı": i, "cinsiyet": meta_te.loc[i, "Gender"], "ırk": meta_te.loc[i, "Race"],
         "gerçek y": int(yte[i]), "g olasılığı": round(float(gp_te[i]), 3),
         "tam model olasılığı": round(float(sig(f(Xte[i:i + 1]))[0]), 3)}
    for nm, v in zip(SEL_NAMES, PXV[n]):
        r[nm] = decode(nm, v)
    rows.append(r)
print(pd.DataFrame(rows).set_index("kişi").T.to_string())

## Adım 1 — Kesin müdahaleci Shapley: kendi kodumuz ve dört aksiyom

Koalisyon `S` için `v(S)`, `S` içindeki oyuncuların değeri açıklanan kişiden, dışındakilerin değeri arka plan satırından alınarak hesaplanır ve arka plan satırları üzerinde ortalaması alınır. 10 oyuncu için 1024 koalisyonun hepsi hesaplanır. Shapley değeri

φᵢ = Σ_{S ⊆ N∖{i}} |S|! (n − |S| − 1)! / n! · [ v(S ∪ {i}) − v(S) ].

Katsayılar bir matriste (`WM`, 1024 × 10) toplanır; φ = WMᵀ v olur. Kütüphane kullanılmaz; yalnızca `numpy`.

### Aksiyom testleri
Testler hem gerçek modelin ızgara tablosu üzerinde hem de bilinen çözümlü oyunlarda yapılır. Geçme ölçütü, en büyük mutlak hatanın 10⁻⁹'dan küçük olmasıdır.

- **Verimlilik:** Σφᵢ = g(x) − E_b[g(b)] (üç kişinin hepsi için).
- **Kukla:** Çıktıyı etkilemeyen bir oyuncunun φ değeri sıfırdır. Modelde iki oyuncunun değeri sabit bir düzeye sabitlenerek kukla yapılır.
- **Simetri:** Değer fonksiyonunda yer değiştirebilen iki oyuncunun φ değeri eşittir. İki oyuncu yer değiştirilerek simetrik hale getirilen modelde, kişinin ve arka planın simetrik olduğu durumda sınanır.
- **Toplamsallık:** φ(g₁ + g₂) = φ(g₁) + φ(g₂). g₁ model, g₂ elle yazılmış doğrusal olmayan bir işlevdir.
- **Birim oyunu (ek):** v(S) = 1 ⇔ T ⊆ S için φ, T içindeki her oyuncuda 1/|T|, diğerlerinde 0'dır (katsayı matrisinin doğrudan sınaması).
- **Doğrusal model (ek):** g = w·z + c için φᵢ = wᵢ (xᵢ − E_b[bᵢ]).

In [ ]:
n = N_PLAYERS
MASKS = ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1).astype(bool)   # (1024, 10): koalisyonlar
SIZE_S = MASKS.sum(axis=1)
wgt = lambda k: factorial(k) * factorial(n - 1 - k) / factorial(n)
WM = np.zeros((2 ** n, n))
for i in range(n):
    inS = MASKS[:, i]
    WM[inS, i] = [wgt(k - 1) for k in SIZE_S[inS]]      # S ∪ {i} koalisyonu: +w(|S|)
    WM[~inS, i] = [-wgt(k) for k in SIZE_S[~inS]]       # S koalisyonu: −w(|S|)

def coalition_values(g, x, B, cell=400_000):
    # V[s, b] = g(x_S, b_(N\S)) : her koalisyon (satır) ve her arka plan satırı (sütun) için
    nB = len(B)
    cs = max(1, cell // nB)
    V = np.empty((2 ** n, nB))
    for c in range(0, 2 ** n, cs):
        M = MASKS[c:c + cs]
        Z = np.where(M[:, None, :], x[None, None, :], B[None, :, :])
        V[c:c + len(M)] = g(Z.reshape(-1, n)).reshape(len(M), nB)
    return V

def shapley_exact(g, x, B):
    # döndürür: φ (10,), taban v(∅) = E_b g(b), g(x) = v(N)
    V = coalition_values(g, x, B)
    phi = WM.T @ V.mean(axis=1)
    return phi, float(V[0].mean()), float(V[-1].mean())

def shapley_rows(g, x, B):
    # her arka plan satırı için ayrı φ (B=1 değerleri); ortalamaları shapley_exact ile aynıdır
    V = coalition_values(g, x, B)
    return (WM.T @ V).T, float(V[0].mean())

# --- testler ---
rng_t = np.random.default_rng(SEED)
Bt = IXtr[rng_t.choice(len(IXtr), 50, replace=False)]
tests = []

# 1) verimlilik
errs = []
for nme, x in PX.items():
    phi, base, fx = shapley_exact(g_ix, x, Bt)
    errs.append(abs(phi.sum() - (fx - base)))
tests.append(("Verimlilik", "3 kişi, 50 satırlık arka plan; Σφ = g(x) − taban", max(errs)))

# 2) kukla: ilk iki oyuncu sabit bir düzeye sabitlenir
dum = [0, 1]
def g_dummy(IX):
    IX2 = IX.copy()
    IX2[..., dum] = 0
    return g_ix(IX2)
errs = []
for nme, x in PX.items():
    phi, _, _ = shapley_exact(g_dummy, x, Bt)
    errs.append(np.abs(phi[dum]).max())
tests.append(("Kukla", f"{SEL_NAMES[0]} ve {SEL_NAMES[1]} modelden çıkarıldı; |φ| = 0", max(errs)))

# 3) simetri: düzey sayıları eşit iki oyuncu yer değiştirilir
pair = None
for a in range(n):
    for b in range(a + 1, n):
        if SHAPE[a] == SHAPE[b]:
            pair = (a, b)
            break
    if pair:
        break
if pair:
    a, b = pair
    def swap(IX):
        IX2 = IX.copy()
        IX2[..., [a, b]] = IX[..., [b, a]]
        return IX2
    g_sym = lambda IX: 0.5 * (g_ix(IX) + g_ix(swap(IX)))
    errs = []
    for nme, x in PX.items():
        xs = x.copy()
        xs[b] = xs[a]
        Bs = np.vstack([Bt, swap(Bt)])
        phi, _, _ = shapley_exact(g_sym, xs, Bs)
        errs.append(abs(phi[a] - phi[b]))
    tests.append(("Simetri", f"{SEL_NAMES[a]} ve {SEL_NAMES[b]} yer değiştirilebilir; φ eşit", max(errs)))
else:
    print("Düzey sayıları eşit iki oyuncu yok; simetri testi atlandı")

# 4) toplamsallık
g2 = lambda IX: 0.3 * IX[..., 0] - 0.2 * IX[..., 1] * IX[..., 2] + 0.05 * IX[..., 3] ** 2 + np.sin(IX[..., 4])
g_sum = lambda IX: g_ix(IX) + g2(IX)
errs = []
for nme, x in PX.items():
    p1, _, _ = shapley_exact(g_ix, x, Bt)
    p2, _, _ = shapley_exact(g2, x, Bt)
    ps, _, _ = shapley_exact(g_sum, x, Bt)
    errs.append(np.abs(ps - (p1 + p2)).max())
tests.append(("Toplamsallık", "φ(g + h) − (φ(g) + φ(h)); h elle yazılmış doğrusal olmayan işlev", max(errs)))

# 5) birim oyunu
T_ = [0, 3, 5]
v_unit = np.array([float(all(MASKS[s, t] for t in T_)) for s in range(2 ** n)])
phi_u = WM.T @ v_unit
true_u = np.array([1 / len(T_) if i in T_ else 0.0 for i in range(n)])
tests.append(("Birim oyunu", "T = {0, 3, 5}; φ = 1/3 (T içinde), 0 (dışında)", np.abs(phi_u - true_u).max()))

# 6) doğrusal model
wl = np.random.default_rng(1).normal(size=n)
g_lin = lambda IX: IX @ wl + 0.7
errs = []
for nme, x in PX.items():
    phi, _, _ = shapley_exact(g_lin, x, Bt)
    errs.append(np.abs(phi - wl * (x - Bt.mean(axis=0))).max())
tests.append(("Doğrusal model", "φᵢ = wᵢ (xᵢ − E[bᵢ])", max(errs)))

axiom = pd.DataFrame(tests, columns=["test", "kurulum", "en büyük mutlak hata"])
axiom["sonuç"] = np.where(axiom["en büyük mutlak hata"] < 1e-9, "GEÇTİ", "KALDI")
axiom["en büyük mutlak hata"] = axiom["en büyük mutlak hata"].map(lambda v: f"{v:.2e}")
print(axiom.to_string(index=False))

## Adım 2 — `shap` kütüphanesiyle karşılaştırma

Aynı arka plan ve aynı oyun üç kütüphane yöntemiyle çözülür:

- **Müdahaleci TreeSHAP** (`TreeExplainer`, `feature_perturbation="interventional"`, `model_output="raw"`): Gerçek model üzerinde çalışır; arka plan satırlarının kalan 14 özniteliği ortalamaya, açıklanan kişinin kalan 14 özniteliği de ortalamaya sabitlenir. Bu 14 özniteliğin φ değeri bu durumda sıfır olmalıdır.
- **ExactExplainer** (`shap.explainers.Exact`, `Independent` masker; arka planın tamamı kullanılır): 10 oyunculu `g` işlevi üzerinde.
- **KernelExplainer** (`nsamples = 2¹⁰`; tüm koalisyonlar sayıldığı için kesin): aynı `g` işlevi üzerinde.

Arka plan, eğitim kümesinden `B_SHAP` = 100 satırlık rastgele bir altkümedir (tohum 42). Kendi kodumuz aynı satırlarla çalışır.

Not: `TreeExplainer`, `data` olarak 100 satırdan fazlası verilirse arka planı uyarı vererek **100 satıra indirir**. Bu yüzden arka plan, `Independent(..., max_samples=satır sayısı)` maskesiyle verilir; böylece kütüphane de, verilen satırların hepsini kullanır.

In [ ]:
rng_b = np.random.default_rng(SEED)
b_idx = rng_b.choice(len(Xtr), B_SHAP, replace=False)
Bv, Bix = Xtr[b_idx][:, SEL], IXtr[b_idx]
B_full = to_full(Bv)

ex_tree = shap.TreeExplainer(bb, data=shap.maskers.Independent(B_full, max_samples=len(B_full)), feature_perturbation="interventional", model_output="raw")
masker = shap.maskers.Independent(Bv, max_samples=len(Bv))
ex_exact = shap.explainers.Exact(g_model, masker)
ex_kern = shap.KernelExplainer(g_model, Bv)

cmp_rows, detail = [], {}
for nme, i in PERSONS.items():
    xv = PXV[nme]
    phi_own, base_own, fx = shapley_exact(g_ix, PX[nme], Bix)

    t = time.time()
    xf = to_full(xv[None, :])
    sv = ex_tree.shap_values(xf)
    sv = np.asarray(sv).reshape(1, -1)[0]
    phi_tree = sv[SEL]
    dt_tree = time.time() - t
    base_tree = float(np.ravel(ex_tree.expected_value)[0])
    oth_max = np.abs(sv[OTH]).max()

    t = time.time()
    e = ex_exact(xv[None, :])
    phi_exact, base_exact = np.asarray(e.values)[0], float(np.ravel(e.base_values)[0])
    dt_exact = time.time() - t

    t = time.time()
    phi_kern = np.asarray(ex_kern.shap_values(xv[None, :], nsamples=2 ** N_PLAYERS, silent=True)).reshape(-1)
    dt_kern = time.time() - t
    base_kern = float(np.ravel(ex_kern.expected_value)[0])

    detail[nme] = pd.DataFrame({"kendi kodumuz": phi_own, "TreeSHAP (müdahaleci)": phi_tree,
                                "ExactExplainer": phi_exact, "KernelExplainer": phi_kern}, index=SEL_NAMES)
    for ad, ph, bs, dt in [("TreeExplainer (müdahaleci)", phi_tree, base_tree, dt_tree),
                           ("ExactExplainer", phi_exact, base_exact, dt_exact),
                           ("KernelExplainer", phi_kern, base_kern, dt_kern)]:
        d = np.abs(ph - phi_own)
        cmp_rows.append({"kişi": nme, "explainer": ad, "en büyük |Δφ|": d.max(), "en büyük farkın özniteliği": SEL_NAMES[int(d.argmax())],
                         "|Δtaban|": abs(bs - base_own), "süre (sn)": round(dt, 2)})
    print(f"{nme}: TreeSHAP'ta sabitlenen 14 özniteliğin en büyük |φ| = {oth_max:.2e} | kendi kodumuzda Σφ − (g(x) − taban) = {phi_own.sum() - (fx - base_own):.2e}")

cmp = pd.DataFrame(cmp_rows)
fmt = cmp.copy()
for c in ["en büyük |Δφ|", "|Δtaban|"]:
    fmt[c] = fmt[c].map(lambda v: f"{v:.2e}")
print("\nKendi kodumuz ile kütüphane arasındaki fark (arka plan: 100 eğitim satırı; çıktı: log-odds):")
print(fmt.to_string(index=False))
mx = cmp.loc[cmp["en büyük |Δφ|"].idxmax()]
print(f"\nEn büyük fark: {mx['en büyük |Δφ|']:.2e} ({mx['explainer']}, {mx['kişi']}, {mx['en büyük farkın özniteliği']})")
print("\nÖrnek (Kişi A): oyuncu başına φ")
print(detail["Kişi A"].round(5).to_string())

## Adım 3 — Arka plan boyutu

Her kişi için arka plan boyutu 1, 10, 100 ve 500 yapılır; her boyutta `N_SEEDS` = 20 tohum, her tohumda eğitim kümesinden yerine koymadan çekilmiş bir rastgele altküme kullanılır. Bir tohumun altkümeleri iç içedir (500'lük çekilişin ilk 1, 10, 100 satırı) ve aynı tohum üç kişi için de aynı satırları verir.

**Referans:** eğitim kümesinin tamamı (18.084 satır) arka plan olarak alınan kesin Shapley değeri (kendi kodumuzla). **Sapma ölçüsü:** göreli sapma ‖φ_B − φ_ref‖₂ / ‖φ_ref‖₂. İki yardımcı ölçü: işaret uyumu (|φ_ref| ≥ 0,01 olan oyuncularda işaretin referansla aynı olma oranı) ve ilk üç öznitelik kümesinin (|φ| büyüklüğüne göre) referansla aynı olma oranı.

Kuramsal karşılaştırma: arka plan satırları kümeden yerine koymadan çekildiği için φ_B, her arka plan satırının tek başına Shapley değerinin (B = 1) örnek ortalamasıdır. Bu durumda beklenen karesel sapma (σ²/n)·(N − n)/(N − 1)'dir (σ² = tek satırlı değerlerin toplam varyansı). Bu çizgi şekilde kesikli olarak gösterilir.

In [ ]:
t0 = time.time()
REF, BASE_REF, VAR_SUM = {}, {}, {}
for nme in PERSONS:
    Phi, base = shapley_rows(g_ix, PX[nme], IXtr)
    REF[nme], BASE_REF[nme] = Phi.mean(axis=0), base
    VAR_SUM[nme] = Phi.var(axis=0, ddof=0).sum()
    chk_phi, chk_base, _ = shapley_exact(g_ix, PX[nme], IXtr)
    assert np.abs(chk_phi - REF[nme]).max() < 1e-12
print(f"Referanslar hazır ({time.time() - t0:.0f} sn). Referans taban (log-odds):", {k: round(v, 3) for k, v in BASE_REF.items()})

Ntr = len(IXtr)
def topk(phi, k=3):
    return set(np.argsort(-np.abs(phi))[:k])

dev_rows = []
for seed in range(N_SEEDS):
    perm = np.random.default_rng(SEED + 1000 + seed).permutation(Ntr)[:max(SIZES)]
    for nme in PERSONS:
        ref = REF[nme]
        big = np.abs(ref) >= EPS_SIGN
        for s in SIZES:
            phi, base, _ = shapley_exact(g_ix, PX[nme], IXtr[perm[:s]])
            dev_rows.append({"kişi": nme, "boyut": s, "tohum": seed,
                             "göreli sapma": np.linalg.norm(phi - ref) / np.linalg.norm(ref),
                             "işaret uyumu": float((np.sign(phi[big]) == np.sign(ref[big])).mean()),
                             "ilk 3 aynı": float(topk(phi) == topk(ref)),
                             "taban sapması": abs(base - BASE_REF[nme])})
dev = pd.DataFrame(dev_rows)
print(f"Boyut deneyi bitti ({time.time() - t0:.0f} sn): {len(dev)} hesap")

summ = dev.groupby(["kişi", "boyut"]).agg(ort=("göreli sapma", "mean"), sd=("göreli sapma", "std"),
                                           min=("göreli sapma", "min"), maks=("göreli sapma", "max"),
                                           işaret_uyumu=("işaret uyumu", "mean"), ilk3_aynı=("ilk 3 aynı", "mean"),
                                           taban_sapması=("taban sapması", "mean")).reset_index()
theo = []
for _, r in summ.iterrows():
    N_, s = Ntr, r["boyut"]
    theo.append(np.sqrt(VAR_SUM[r["kişi"]] / s * (N_ - s) / (N_ - 1)) / np.linalg.norm(REF[r["kişi"]]))
summ["kuramsal"] = theo
print(f"Göreli sapma ({N_SEEDS} tohum; referans: tüm eğitim kümesi):")
print(summ.round(4).to_string(index=False))

In [ ]:
styles = {"Kişi A": ("-", "o"), "Kişi B": ("--", "s"), "Kişi C": (":", "^")}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
ax = axes[0]
for nme in PERSONS:
    ls, mk = styles[nme]
    d = summ[summ["kişi"] == nme]
    ax.errorbar(d["boyut"], d["ort"], yerr=d["sd"], color="black", ls=ls, marker=mk, capsize=3, label=f"{nme} (ortalama ± sd)")
    ax.plot(d["boyut"], d["kuramsal"], color="0.55", ls=ls, lw=1.2)
    sub = dev[dev["kişi"] == nme]
    ax.scatter(sub["boyut"] * (1 + 0.04 * list(PERSONS).index(nme)), sub["göreli sapma"], s=6, color="0.6", alpha=0.5)
ax.plot([], [], color="0.55", lw=1.2, label="kuramsal (örnekleme hatası)")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(SIZES); ax.set_xticklabels(SIZES)
ax.set_xlabel("arka plan boyutu (satır)")
ax.set_ylabel("göreli sapma  ‖φ_B − φ_ref‖ / ‖φ_ref‖")
ax.set_title(f"Arka plan boyutuna göre sapma ({N_SEEDS} tohum)")
ax.legend(fontsize=8)
ax = axes[1]
for nme in PERSONS:
    ls, mk = styles[nme]
    d = summ[summ["kişi"] == nme]
    ax.plot(d["boyut"], d["işaret_uyumu"], color="black", ls=ls, marker=mk, label=f"{nme}: işaret uyumu")
    ax.plot(d["boyut"], d["ilk3_aynı"], color="0.55", ls=ls, marker=mk, label=f"{nme}: ilk 3 aynı")
ax.set_xscale("log"); ax.set_xticks(SIZES); ax.set_xticklabels(SIZES)
ax.set_ylim(0, 1.05)
ax.set_xlabel("arka plan boyutu (satır)")
ax.set_ylabel("oran")
ax.set_title("İşaret uyumu ve ilk üç özniteliğin doğruluğu")
ax.legend(fontsize=7, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.17))
fig.tight_layout()
savefig(fig, "h5_01_arka_plan_boyutu.png")
plt.show()

### 3.2 Arka plan bileşimi
Boyut sabit tutulup bileşim değiştirilir: 200 satırlık arka planın `y = 1` (3 yıl içinde yeniden tutuklanan) oranı 0, 0,25, 0,5, 0,75 ve 1 yapılır; her oranda 20 tohum. Shapley değeri arka plan dağılımının doğrusal bir işlevi olduğu için φ'nin bu orana göre doğrusala yakın değişmesi beklenir. Tabloda ortalama (ve parantezde tohumlar arası standart sapma) verilir.

In [ ]:
pos, neg = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
comp_rows = []
for q in COMP_FRACS:
    kq = int(round(q * COMP_SIZE))
    for seed in range(N_COMP_SEEDS):
        rg = np.random.default_rng(SEED + 5000 + seed)
        idx = np.concatenate([rg.choice(pos, kq, replace=False), rg.choice(neg, COMP_SIZE - kq, replace=False)])
        for nme in PERSONS:
            phi, base, _ = shapley_exact(g_ix, PX[nme], IXtr[idx])
            comp_rows.append({"kişi": nme, "y=1 oranı": q, "tohum": seed, "taban": base, **dict(zip(SEL_NAMES, phi))})
comp = pd.DataFrame(comp_rows)
cols_show = {}
for nme in PERSONS:
    top4 = list(pd.Series(REF[nme], index=SEL_NAMES).abs().sort_values(ascending=False).index[:4])
    cols_show[nme] = top4
    g_ = comp[comp["kişi"] == nme].groupby("y=1 oranı")
    m_, s_ = g_[["taban"] + top4].mean(), g_[["taban"] + top4].std()
    tab = m_.map(lambda v: f"{v:+.3f}") + " (" + s_.map(lambda v: f"{v:.3f}") + ")"
    print(f"\n{nme}: arka plan bileşimine göre taban ve en büyük dört |φ| (log-odds; ortalama (sd), {N_COMP_SEEDS} tohum, boyut {COMP_SIZE})")
    print(tab.to_string())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
mks = ["o", "s", "^", "D"]
for ax, nme in zip(axes, PERSONS):
    g_ = comp[comp["kişi"] == nme].groupby("y=1 oranı")
    for mk, f_ in zip(mks, cols_show[nme]):
        ax.errorbar(COMP_FRACS, g_[f_].mean(), yerr=g_[f_].std(), color="black", marker=mk, capsize=3, ls="-", lw=1, label=f_.replace("Prior_", "").replace("_Episodes", ""))
    ax.axhline(0, color="0.6", lw=0.6)
    ax.set_xlabel("arka plandaki yeniden tutuklanan oranı")
    ax.set_ylabel("φ (log-odds)")
    ax.set_title(nme)
    ax.legend(fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.2))
fig.tight_layout()
savefig(fig, "h5_02_bilesim.png")
plt.show()

## Adım 4 — Anlamlı referans grupları

Aynı üç kişi için altı referans grubu tanımlanır (hepsi **eğitim kümesinden**, grubun tamamı arka plan olarak kullanılır; rastgelelik yoktur):

- **R1 — Tüm eğitim kümesi:** Varsayılan arka plan.
- **R2 — Yeniden tutuklanmayanlar (y = 0):** "Olumlu sonuçlanan" kişiler; açıklama, kişinin bu kişilerden farkını gösterir.
- **R3 — Aynı yaş grubu:** Yaş sabit kalır; yaş dışındaki farklar açıklanır.
- **R4 — Aynı cinsiyet:** `Gender` modele girmez, ancak bu veri kümesinde `Gang_Affiliated = Unknown` düzeyi tüm kadınlarla örtüşür (kadınların hepsi `Unknown`, `Unknown` olanların hepsi kadındır); bu grup bu durumun açıklamaya etkisini gösterir.
- **R5 — Aynı hapis süresi grubu.**
- **R6 — Modelin düşük riskli bulduğu kişiler:** Eğitim kümesinde `g` değeri en düşük %25'te olanlar.

Her grup için taban `v(∅)` (log-odds ve ortalama risk olasılığı), kişinin `g(x)` değeri ve Shapley değerleri verilir. Gruplar arasında φ'nin işareti değişen (bir grupta > +0,01, bir grupta < −0,01) öznitelikler `*` ile işaretlenir. Bir grubun üyeleri kişiyle aynı değere sahipse, o özniteliğin φ değeri sıfırdır (kukla oyuncu).

In [ ]:
age_j, py_j = SEL_NAMES.index("Age_at_Release"), SEL_NAMES.index("Prison_Years")
g_tr_q25 = np.quantile(g_tr, 0.25)
gender_tr = meta_tr["Gender"].to_numpy()

def groups_for(nme, i):
    x = PX[nme]
    return {
        "R1 tüm eğitim": np.ones(Ntr, bool),
        "R2 y=0": ytr == 0,
        "R3 aynı yaş": IXtr[:, age_j] == x[age_j],
        "R4 aynı cinsiyet": gender_tr == meta_te.loc[i, "Gender"],
        "R5 aynı hapis süresi": IXtr[:, py_j] == x[py_j],
        "R6 düşük risk (alt %25)": g_tr <= g_tr_q25,
    }

REFTAB, FLAGS = {}, {}
for nme, i in PERSONS.items():
    cols, info = {}, {}
    for gname, mask in groups_for(nme, i).items():
        phi, base, fx = shapley_exact(g_ix, PX[nme], IXtr[mask])
        cols[gname] = phi
        info[gname] = (int(mask.sum()), base, float(sig(g_tr[mask]).mean()), fx)
    P = pd.DataFrame(cols, index=SEL_NAMES)
    flag = ((P > EPS_SIGN).any(axis=1)) & ((P < -EPS_SIGN).any(axis=1))
    FLAGS[nme] = list(P.index[flag])
    out = P.map(lambda v: f"{round(v, 3) + 0.0:+.3f}").copy()
    out.index = [nm + (" *" if flag[nm] else "") for nm in P.index]
    extra = pd.DataFrame({gname: [str(info[gname][0]), f"{info[gname][1]:+.3f}", f"{info[gname][2]:.3f}", f"{info[gname][3]:+.3f}", f"{P[gname].sum():+.3f}"] for gname in cols},
                         index=["arka plan n", "taban v(∅) (log-odds)", "taban: ort. risk olasılığı", "g(x) (log-odds)", "Σφ  (= g(x) − taban)"])
    REFTAB[nme] = P
    print(f"\n{nme} (test satırı {i}; cinsiyet {meta_te.loc[i, 'Gender']}; g olasılığı {gp_te[i]:.3f}) — değer fonksiyonu: E_b[g(x_S, b)], çıktı: log-odds; explainer: kendi kesin müdahaleci Shapley kodumuz")
    print(pd.concat([out, extra]).to_string())
    print("İşaret değiştiren öznitelikler (*):", FLAGS[nme] if FLAGS[nme] else "yok")
    print("Kukla (|φ| < 1e-9) olan öznitelikler:", {gname: [nm for nm in SEL_NAMES if abs(P.loc[nm, gname]) < 1e-9] for gname in cols if (P[gname].abs() < 1e-9).any()})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.2))
mk6 = ["o", "s", "^", "D", "v", "P"]
for ax, nme in zip(axes, PERSONS):
    P = REFTAB[nme]
    ypos = np.arange(len(SEL_NAMES))[::-1]
    for k, gname in enumerate(P.columns):
        ax.scatter(P[gname].values, ypos + (k - 2.5) * 0.09, marker=mk6[k], s=30, facecolor="black" if k % 2 == 0 else "white", edgecolor="black", label=gname)
    ax.axvline(0, color="0.5", lw=0.8)
    ax.set_yticks(ypos)
    ax.set_yticklabels([nm + (" *" if nm in FLAGS[nme] else "") for nm in SEL_NAMES], fontsize=8)
    ax.set_xlabel("φ (log-odds)")
    ax.set_title(nme)
axes[0].legend(fontsize=7, loc="lower right")
fig.suptitle("Referans grubuna göre Shapley değerleri (* = işaret değiştiren öznitelik)")
fig.tight_layout()
savefig(fig, "h5_03_referans_gruplari.png")
plt.show()

## Adım 5 — Yol bağımlı ve müdahaleci TreeSHAP

Model bir ağaç topluluğu olduğu için (sinir ağı değil) bu adımda TreeSHAP'ın iki sürümü karşılaştırılır; integrated gradients kullanılmaz.

- **Yol bağımlı** (`tree_path_dependent`): Arka plan verisi gerekmez; bir düğümde eksik özniteliğin hangi dala gideceği, eğitimde o düğümden geçen örnek sayılarıyla (kapsam) belirlenir. Bu, koşullu beklentiyi yaklaştırır.
- **Müdahaleci** (`interventional`): Eksik öznitelik arka plan satırlarından alınır (marjinal beklenti). Arka plan: eğitim kümesinden `B_PATH` = 2000 rastgele satır.

Karşılaştırma **24 özniteliğin hepsi** için, kişilerin özgün değerleriyle yapılır (kalan 14 öznitelik burada sabitlenmez). Raporlanan: iki sürümün tabanı, tamlık hatası (Σφ + taban − f(x)), φ vektörleri arasındaki Spearman korelasyonu, en büyük fark ve işareti ters olan öznitelikler (|φ| > 0,01).

In [ ]:
rg = np.random.default_rng(SEED)
bg_idx = rg.choice(len(Xtr), B_PATH, replace=False)
t = time.time()
ex_pd = shap.TreeExplainer(bb, feature_perturbation="tree_path_dependent", model_output="raw")
ex_iv = shap.TreeExplainer(bb, data=shap.maskers.Independent(Xtr[bg_idx], max_samples=B_PATH), feature_perturbation="interventional", model_output="raw")
print(f"Explainer'lar hazır ({time.time() - t:.1f} sn)")

SV, summ5 = {}, []
for nme, i in PERSONS.items():
    x = Xte[i:i + 1]
    t = time.time(); s_pd = np.asarray(ex_pd.shap_values(x)).reshape(-1); t_pd = time.time() - t
    t = time.time(); s_iv = np.asarray(ex_iv.shap_values(x)).reshape(-1); t_iv = time.time() - t
    b_pd, b_iv = float(np.ravel(ex_pd.expected_value)[0]), float(np.ravel(ex_iv.expected_value)[0])
    fx = float(f(x)[0])
    d = s_iv - s_pd
    sign_bad = [FEATS[j] for j in range(len(FEATS)) if max(abs(s_iv[j]), abs(s_pd[j])) > EPS_SIGN and np.sign(s_iv[j]) != np.sign(s_pd[j])]
    SV[nme] = pd.DataFrame({"yol bağımlı": s_pd, "müdahaleci": s_iv, "fark (müd. − yol)": d}, index=FEATS)
    summ5.append({"kişi": nme, "f(x) log-odds": fx, "taban (yol)": b_pd, "taban (müd.)": b_iv,
                  "tamlık hatası (yol)": abs(s_pd.sum() + b_pd - fx), "tamlık hatası (müd.)": abs(s_iv.sum() + b_iv - fx),
                  "Spearman": spearmanr(s_pd, s_iv)[0], "en büyük |fark|": np.abs(d).max(),
                  "en büyük farkın özniteliği": FEATS[int(np.abs(d).argmax())], "toplam |fark|": np.abs(d).sum(),
                  "işareti ters": len(sign_bad), "süre yol/müd. (sn)": f"{t_pd:.2f}/{t_iv:.2f}"})
    print(f"\n{nme}: işareti ters olan öznitelikler (|φ| > {EPS_SIGN}):", sign_bad if sign_bad else "yok")
s5 = pd.DataFrame(summ5).set_index("kişi")
for c in ["tamlık hatası (yol)", "tamlık hatası (müd.)"]:
    s5[c] = s5[c].map(lambda v: f"{v:.2e}")
print("\nÖzet (log-odds):")
print(s5.round(4).T.to_string())

for nme in PERSONS:
    T = SV[nme]
    T = T.reindex(T[["yol bağımlı", "müdahaleci"]].abs().max(axis=1).sort_values(ascending=False).index)
    print(f"\n{nme}: φ değerleri, en büyük 12 |φ| (explainer: TreeExplainer; değer fonksiyonu: yol bağımlı = kapsam ağırlıklı koşullu, müdahaleci = marjinal, {B_PATH} satırlık arka plan; çıktı log-odds)")
    print(T.head(12).round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, nme in zip(axes, PERSONS):
    T = SV[nme]
    ax.scatter(T["yol bağımlı"], T["müdahaleci"], s=22, color="0.25")
    lim = 1.1 * max(T[["yol bağımlı", "müdahaleci"]].abs().max())
    ax.plot([-lim, lim], [-lim, lim], color="0.5", lw=0.8)
    ax.axhline(0, color="0.8", lw=0.6); ax.axvline(0, color="0.8", lw=0.6)
    for fn in T["fark (müd. − yol)"].abs().sort_values(ascending=False).index[:3]:
        ax.annotate(fn.replace("Prior_", "").replace("_Episodes", ""), (T.loc[fn, "yol bağımlı"], T.loc[fn, "müdahaleci"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("φ yol bağımlı"); ax.set_ylabel("φ müdahaleci")
    ax.set_title(nme)
fig.suptitle("Yol bağımlı ve müdahaleci TreeSHAP (24 öznitelik, log-odds)")
fig.tight_layout()
savefig(fig, "h5_04_yol_bagimli_mudahaleci.png")
plt.show()

## Çıktılar

Notebook, şekilleri `h5_01_…` ile `h5_04_…` adlı PNG dosyaları olarak çalışma klasörüne kaydeder. Rapor için gerekli sayılar yukarıdaki hücrelerin çıktılarında yazdırılmıştır: aksiyom tablosu (Adım 1), kütüphane karşılaştırması (Adım 2), boyut ve bileşim deneyleri (Adım 3), referans grubu tabloları (Adım 4), yol bağımlı / müdahaleci karşılaştırma (Adım 5).